# 09 - Secuencias: RNN y LSTM

**AI sin humo** - Notas personales para entender deep learning desde cero.

Hasta ahora, todo lo que vimos (perceptrón, shallow networks, deep networks, backprop, regularización) asumía una cosa: **el input tiene tamaño fijo**. Le pasás un vector de dimensión $D$, la red lo procesa, y te da un output. Listo.

Pero pensá en el mundo real. El lenguaje no tiene longitud fija. Una frase puede tener 3 palabras o 300. Un audio puede durar 2 segundos o 2 horas. Una serie temporal de la bolsa tiene datos de cada día durante años. **¿Cómo hacemos para que una red neuronal procese secuencias de longitud variable?**

Este notebook responde esa pregunta. Vamos a ver cómo las RNN (Recurrent Neural Networks) resuelven el problema con una idea elegante: **procesar un elemento a la vez manteniendo memoria de lo que ya vimos**. Después vamos a ver por qué las RNN vanilla tienen problemas serios, y cómo las LSTM (Long Short-Term Memory) los resuelven con un diseño brillante de gates.

---

## Contenido

1. [El problema de las secuencias](#secuencias)
2. [Recurrent Neural Networks (RNN)](#rnn)
3. [Problemas de las RNN: vanishing gradients](#problemas-rnn)
4. [LSTM (Long Short-Term Memory)](#lstm)
5. [Intuición de diseño de arquitecturas](#intuicion-diseno)
6. [RNNs son Turing-Complete](#turing-complete)
7. [Deep RNN (RNNs apiladas)](#deep-rnn)
8. [El bottleneck del hidden state](#bottleneck)
9. [Seq2Seq (Encoder-Decoder)](#seq2seq)
10. [Resumen](#resumen)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

torch.manual_seed(42)
np.random.seed(42)

---

<a id='secuencias'></a>
## 1. El problema de las secuencias

### Las redes normales (MLP) asumen input fijo

Todo lo que vimos hasta ahora funciona así: tenés un vector de entrada $\mathbf{x} \in \mathbb{R}^D$, con $D$ fijo. La red tiene una primera capa con una matriz de pesos $\mathbf{W}_1 \in \mathbb{R}^{H \times D}$ que espera exactamente $D$ features de entrada.

Esto funciona perfecto para:
- Clasificar imágenes de tamaño fijo (ej: 28×28 = 784 pixels)
- Predecir precios a partir de features tabulares (ej: metros cuadrados, ubicación, etc.)
- Cualquier problema donde el input siempre tiene la misma dimensión

### ¿Qué pasa con secuencias de longitud variable?

Ahora pensá en estos problemas:
- **Texto**: "hola" tiene 4 letras, "el rápido zorro marrón salta sobre el perro perezoso" tiene muchas más
- **Audio**: una grabación de 1 segundo vs una de 10 minutos
- **Series temporales**: datos bursátiles de 1 mes vs 10 años
- **Video**: un clip de 3 segundos vs una película de 2 horas

El **largo de la secuencia varía** de ejemplo a ejemplo. No podemos simplemente definir una red con un input de tamaño fijo.

### "Pero podríamos hacer padding..." — No tan rápido

La primera idea que se te puede ocurrir es: "bueno, definamos un tamaño máximo, rellenemos con ceros (padding) las secuencias más cortas, y listo. Flattenizamos todo y se lo pasamos a un MLP".

Suena razonable, pero tiene **problemas fundamentales**:

In [ ]:
# Demo: why padding + MLP doesn't work for sequences

# Suppose we have a vocabulary of 5 tokens: {a, b, c, d, e} -> {0, 1, 2, 3, 4}
# And we want to process sequences of variable length

# Sequence 1: [a, b, c]    -> length 3
# Sequence 2: [a, b, c, d, e] -> length 5

max_len = 5
vocab_size = 5

# Naive approach: pad to max_len and one-hot encode each position
# Then flatten into a single vector

def naive_encode(sequence, max_len, vocab_size):
    """Pad sequence, one-hot encode, and flatten for MLP."""
    padded = sequence + [0] * (max_len - len(sequence))  # pad with 0s
    one_hot = np.zeros((max_len, vocab_size))
    for i, token in enumerate(padded):
        one_hot[i, token] = 1.0
    return one_hot.flatten()  # shape: (max_len * vocab_size,)

seq1 = [0, 1, 2]        # "a b c"
seq2 = [0, 1, 2, 3, 4]  # "a b c d e"

encoded1 = naive_encode(seq1, max_len, vocab_size)
encoded2 = naive_encode(seq2, max_len, vocab_size)

print(f"Sequence 1: {seq1}")
print(f"  Encoded shape: {encoded1.shape}")
print(f"  Encoded: {encoded1}")
print()
print(f"Sequence 2: {seq2}")
print(f"  Encoded shape: {encoded2.shape}")
print(f"  Encoded: {encoded2}")

### Los 3 problemas fatales del enfoque naive

**Problema 1: No reconoce tokens individuales**

Cuando flattenizás, el token "a" en la posición 0 y el token "a" en la posición 3 son features completamente diferentes para la red. El MLP tiene pesos separados para cada posición. No puede "entender" que es el mismo token en distinta posición. Si aprendió que "a" es importante en la posición 0, tiene que **re-aprender** eso para la posición 3 desde cero.

**Problema 2: Los pesos no se adaptan dinámicamente al contexto**

En una secuencia, el significado de cada elemento depende de lo que vino antes. "banco" significa algo distinto en "me senté en el banco" que en "fui al banco a sacar plata". Un MLP no tiene mecanismo para que la interpretación de un token dependa de los tokens anteriores de forma dinámica. Cada posición se procesa con pesos fijos que no saben nada del contexto.

**Problema 3: No escala**

Si querés manejar secuencias de hasta 1000 tokens con vocabulario de 50,000 (típico en NLP), tu vector de entrada tendría dimensión 1000 × 50,000 = **50 millones**. La primera capa de un MLP necesitaría una matriz de pesos con 50 millones de columnas. Completamente impracticable.

$$\text{Dimensión input} = T_{max} \times V = 1000 \times 50000 = 50,000,000$$

Necesitamos algo fundamentalmente diferente. Necesitamos una arquitectura que:
1. Procese **un token a la vez** (no todos juntos)
2. **Comparta pesos** entre todos los pasos temporales
3. Mantenga alguna forma de **memoria** de lo que ya procesó

Esa arquitectura es la **Recurrent Neural Network (RNN)**.

---

<a id='rnn'></a>
## 2. Recurrent Neural Networks (RNN)

### Cómo procesamos secuencias nosotros

Pensá en cómo leés una oración. No mirás todas las palabras al mismo tiempo y las procesás en paralelo (bueno, más o menos, pero simplificamos). Lo que hacés es:

1. Leés la primera palabra → actualizás tu entendimiento
2. Leés la segunda palabra → la combinás con lo que ya entendías → actualizás
3. Leés la tercera palabra → la combinás con tu entendimiento acumulado → actualizás
4. ... y así sucesivamente

En cada paso, hacés dos cosas:
- Tomás **input nuevo** (la palabra actual)
- Lo combinás con tu **memoria** de todo lo anterior
- Generás un **nuevo estado mental** actualizado

Una RNN hace exactamente esto, pero con matrices de pesos y funciones de activación.

### El hidden state como memoria

La idea clave de una RNN es el **hidden state** $\mathbf{h}_t$: un vector que representa la "memoria" de todo lo que la red procesó hasta el paso $t$.

En cada paso temporal $t$, la RNN:
1. Recibe el input actual $\mathbf{x}_t$
2. Recibe el hidden state anterior $\mathbf{h}_{t-1}$ (la memoria)
3. Combina ambos para producir un nuevo hidden state $\mathbf{h}_t$

### La ecuación fundamental de una RNN

$$\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \cdot \mathbf{x}_t + \mathbf{W}_{hh} \cdot \mathbf{h}_{t-1} + \mathbf{b}_h)$$

Donde:
- $\mathbf{x}_t \in \mathbb{R}^D$: input en el paso $t$ (ej: embedding del token actual)
- $\mathbf{h}_{t-1} \in \mathbb{R}^H$: hidden state del paso anterior (memoria)
- $\mathbf{W}_{xh} \in \mathbb{R}^{H \times D}$: pesos que transforman el input
- $\mathbf{W}_{hh} \in \mathbb{R}^{H \times H}$: pesos que transforman el hidden state anterior
- $\mathbf{b}_h \in \mathbb{R}^H$: bias
- $\tanh$: función de activación (squashes output entre -1 y 1)

Y si necesitamos un output en cada paso:

$$\mathbf{y}_t = \mathbf{W}_{hy} \cdot \mathbf{h}_t + \mathbf{b}_y$$

### Lo fundamental: weight sharing

Mirá las matrices $\mathbf{W}_{xh}$, $\mathbf{W}_{hh}$, $\mathbf{W}_{hy}$: son **las mismas** en todos los pasos temporales. No hay un $\mathbf{W}_{xh}^{(1)}$ para el paso 1 y un $\mathbf{W}_{xh}^{(2)}$ para el paso 2. Son exactamente los mismos pesos.

Esto es lo que resuelve los tres problemas del enfoque naive:
1. **Reconoce tokens**: los mismos pesos procesan cada token, así que lo que aprende sobre un token aplica en cualquier posición
2. **Contexto dinámico**: el hidden state $\mathbf{h}_{t-1}$ lleva información de todos los pasos anteriores, así que el procesamiento del token actual depende del contexto
3. **Escala**: el número de parámetros **no depende de la longitud de la secuencia**, solo de $D$ y $H$

In [ ]:
# Step-by-step RNN forward pass — manually, to understand exactly what happens

# Dimensions
input_dim = 4    # D: dimension of each input token
hidden_dim = 3   # H: dimension of hidden state
seq_len = 3      # T: number of time steps

# Create input sequence: 3 tokens, each of dimension 4
# Think of this as 3 word embeddings
x = torch.randn(seq_len, input_dim)
print("Input sequence shape:", x.shape)
print("Token 0 (x_0):", x[0])
print("Token 1 (x_1):", x[1])
print("Token 2 (x_2):", x[2])
print()

# RNN parameters (shared across all time steps!)
W_xh = torch.randn(hidden_dim, input_dim)   # H x D
W_hh = torch.randn(hidden_dim, hidden_dim)  # H x H
b_h = torch.zeros(hidden_dim)                # H

print(f"W_xh shape: {W_xh.shape}  (transforms input -> hidden)")
print(f"W_hh shape: {W_hh.shape}  (transforms prev hidden -> hidden)")
print(f"b_h shape:  {b_h.shape}")
print()

# Initial hidden state (zeros)
h = torch.zeros(hidden_dim)
print(f"Initial hidden state h_0: {h}")
print("=" * 60)

# Process each time step
all_hidden = []
for t in range(seq_len):
    print(f"\n--- Time step t={t} ---")
    
    # Get current input
    x_t = x[t]
    print(f"x_{t} = {x_t.numpy().round(3)}")
    print(f"h_prev = {h.numpy().round(3)}")
    
    # Apply RNN equation: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    input_contribution = W_xh @ x_t
    hidden_contribution = W_hh @ h
    pre_activation = input_contribution + hidden_contribution + b_h
    h = torch.tanh(pre_activation)
    
    print(f"W_xh @ x_{t} = {input_contribution.detach().numpy().round(3)}  (input contribution)")
    print(f"W_hh @ h_prev = {hidden_contribution.detach().numpy().round(3)}  (memory contribution)")
    print(f"h_{t} = tanh(sum) = {h.detach().numpy().round(3)}")
    
    all_hidden.append(h.clone())

print("\n" + "=" * 60)
print("\nFinal hidden state (carries info about ALL tokens):")
print(f"h_final = {h.detach().numpy().round(3)}")
print(f"\nThis single vector summarizes the entire sequence!")

In [ ]:
# Now let's verify our manual computation matches PyTorch's nn.RNN

rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, batch_first=False, bias=True)

# Copy our weights into PyTorch's RNN
with torch.no_grad():
    rnn.weight_ih_l0.copy_(W_xh)
    rnn.weight_hh_l0.copy_(W_hh)
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.copy_(b_h)

# Forward pass
h0 = torch.zeros(1, hidden_dim)  # (num_layers, hidden_dim)
output, h_final = rnn(x.unsqueeze(1), h0.unsqueeze(0))  # add batch dim

print("PyTorch RNN output (hidden at each step):")
for t in range(seq_len):
    manual = all_hidden[t].detach().numpy().round(3)
    pytorch = output[t, 0].detach().numpy().round(3)
    print(f"  t={t}: manual={manual}, pytorch={pytorch}, match={np.allclose(manual, pytorch, atol=1e-5)}")

print(f"\nFinal hidden state matches: {np.allclose(h.detach().numpy(), h_final.squeeze().detach().numpy(), atol=1e-5)}")

### Unrolling: cómo se ve una RNN "desplegada" en el tiempo

Cuando "desplegamos" (unroll) una RNN en el tiempo, se ve como una cadena de copias de la misma red, donde cada copia le pasa su hidden state a la siguiente:

```
    x_0           x_1           x_2
     │             │             │
     ▼             ▼             ▼
┌─────────┐   ┌─────────┐   ┌─────────┐
│   RNN   │──▶│   RNN   │──▶│   RNN   │
│  Cell   │ h0│  Cell   │ h1│  Cell   │
└─────────┘   └─────────┘   └─────────┘
     │             │             │
     ▼             ▼             ▼
    y_0           y_1           y_2
```

Pero recordá: **todas las cajas "RNN Cell" son la misma función con los mismos pesos**. No son redes separadas. Es la misma red aplicada repetidamente.

Esto es clave para entender backpropagation en RNNs: cuando hacemos backprop, los gradientes fluyen hacia atrás a través de toda la cadena. Esto se llama **Backpropagation Through Time (BPTT)**, y es la fuente de los problemas que veremos a continuación.

---

<a id='problemas-rnn'></a>
## 3. Problemas de las RNN: Vanishing Gradients

### El problema fundamental

Las RNN vanilla tienen un problema devastador: **no pueden recordar información de hace muchos pasos**.

¿Por qué? Por cómo fluyen los gradientes durante backpropagation through time.

Recordá la ecuación:
$$\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \cdot \mathbf{x}_t + \mathbf{W}_{hh} \cdot \mathbf{h}_{t-1} + \mathbf{b})$$

Cuando hacemos backprop, necesitamos calcular $\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}}$. Esto involucra:

$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \text{diag}(1 - \mathbf{h}_t^2) \cdot \mathbf{W}_{hh}$$

El primer término, $\text{diag}(1 - \mathbf{h}_t^2)$, es la derivada de tanh, que está entre 0 y 1.

Para que el gradiente llegue desde el paso $T$ hasta el paso $1$, tiene que pasar por **todos** los pasos intermedios:

$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_1} = \prod_{t=2}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \prod_{t=2}^{T} \text{diag}(1 - \mathbf{h}_t^2) \cdot \mathbf{W}_{hh}$$

¿Ves el problema? Estamos **multiplicando la misma matriz $\mathbf{W}_{hh}$ muchas veces**. Esto es como elevar una matriz a una potencia.

### Qué pasa con multiplicaciones repetidas

Pensalo en una dimensión para simplificar. Si multiplicás un número por 0.9 repetidamente:
- $0.9^{10} = 0.35$
- $0.9^{50} = 0.005$
- $0.9^{100} = 0.000027$ → básicamente cero

Y si multiplicás por 1.1:
- $1.1^{10} = 2.6$
- $1.1^{50} = 117$
- $1.1^{100} = 13,781$ → explota

Con matrices pasa lo mismo pero depende de los eigenvalues:
- Si el eigenvalue más grande es < 1: los gradientes **desaparecen** (vanishing)
- Si el eigenvalue más grande es > 1: los gradientes **explotan** (exploding)

El exploding gradient se puede manejar con **gradient clipping** (simplemente cortás el gradiente si es muy grande). Pero el vanishing gradient es mucho más insidioso: la información simplemente **desaparece** y no hay nada que clippear.

In [ ]:
# Demo: vanishing gradients in RNN
# We'll track the gradient norm at each time step

def track_gradients_rnn(seq_len=30, hidden_dim=32, input_dim=16):
    """Run an RNN and track gradient norms at each time step."""
    
    rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, 
                 batch_first=True, nonlinearity='tanh')
    
    # Create a random input sequence
    x = torch.randn(1, seq_len, input_dim, requires_grad=True)
    h0 = torch.zeros(1, 1, hidden_dim)
    
    # Forward pass — collect hidden states
    output, _ = rnn(x, h0)
    
    # Use loss only at the LAST time step (common scenario)
    # This means gradients have to travel back through ALL time steps
    loss = output[0, -1, :].sum()  # sum of last hidden state
    loss.backward()
    
    # Compute gradient of loss w.r.t. input at each time step
    grad_norms = []
    for t in range(seq_len):
        grad_norm = x.grad[0, t, :].norm().item()
        grad_norms.append(grad_norm)
    
    return grad_norms

# Run multiple times and average
all_norms = []
for _ in range(50):
    torch.manual_seed(_)
    norms = track_gradients_rnn(seq_len=40)
    all_norms.append(norms)

avg_norms = np.mean(all_norms, axis=0)
std_norms = np.std(all_norms, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax = axes[0]
ax.fill_between(range(40), avg_norms - std_norms, avg_norms + std_norms, alpha=0.3, color='C0')
ax.plot(avg_norms, 'o-', color='C0', markersize=4)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Gradient norm (w.r.t. input)', fontsize=12)
ax.set_title('Vanishing Gradients in RNN (linear scale)', fontsize=13)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.annotate('Loss computed\nhere (t=39)', xy=(39, avg_norms[-1]),
            xytext=(30, avg_norms[-1] * 1.5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')
ax.annotate('Almost no gradient\nreaches early steps!', xy=(5, avg_norms[5]),
            xytext=(5, avg_norms[-1] * 0.8),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

# Log scale — shows the exponential decay more clearly
ax = axes[1]
ax.semilogy(avg_norms, 'o-', color='C1', markersize=4)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Gradient norm (log scale)', fontsize=12)
ax.set_title('Vanishing Gradients in RNN (log scale)', fontsize=13)
ax.annotate('Exponential decay!', xy=(20, avg_norms[20]),
            xytext=(8, avg_norms[5] * 0.5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')

plt.tight_layout()
plt.show()

print(f"Gradient norm at t=39 (last step):  {avg_norms[-1]:.6f}")
print(f"Gradient norm at t=20 (mid):        {avg_norms[20]:.6f}")
print(f"Gradient norm at t=0  (first step): {avg_norms[0]:.6f}")
print(f"\nRatio last/first: {avg_norms[-1] / avg_norms[0]:.1f}x")
print("\nThe first time steps barely get any gradient!")
print("This means the RNN can't learn to use info from early in the sequence.")

### Consecuencias prácticas

El vanishing gradient tiene una consecuencia directa y brutal: **la RNN no puede aprender dependencias de largo plazo**.

Imaginá esta tarea de completar texto:
- "Crecí en Francia. Estudié en París. Años después, cuando me pidieron que tradujera un documento, lo hice al ____"

La respuesta correcta es "francés", pero esa información ("Crecí en Francia") está **muchos pasos atrás** en la secuencia. Los gradientes que deberían enseñarle a la RNN a conectar "Francia" con "francés" se desvanecen antes de llegar a esa parte de la secuencia.

Los últimos tokens reciben buen gradiente. Los primeros, casi nada. La RNN efectivamente solo tiene "memoria de corto plazo".

**Necesitamos una forma de que la información fluya a través de muchos pasos sin degradarse.** Eso es exactamente lo que hace la LSTM.

---

<a id='lstm'></a>
## 4. LSTM (Long Short-Term Memory)

### La idea central: dos tipos de memoria

La RNN vanilla tiene un solo vector de memoria: el hidden state $\mathbf{h}_t$. El problema es que este vector tiene que hacer todo: recordar el pasado, procesar el presente, y preparar el output. Demasiada responsabilidad para un solo vector.

La LSTM (Hochreiter & Schmidhuber, 1997) introduce una segunda memoria: el **cell state** $\mathbf{c}_t$.

Ahora tenemos dos memorias con roles distintos:

| Memoria | Símbolo | Rol | Analogía |
|:--------|:--------|:----|:---------|
| **Hidden state** | $\mathbf{h}_t$ | Memoria de trabajo, rápida, output actual | Tu RAM — lo que estás pensando ahora mismo |
| **Cell state** | $\mathbf{c}_t$ | Memoria de largo plazo, protegida, estable | Tu disco duro — almacenamiento persistente |

La genialidad del diseño es que la información en el cell state fluye a través del tiempo con **transformaciones lineales controladas**. No pasa por tanh en cada paso (como en la RNN vanilla), así que no se degrada exponencialmente. Es como una "autopista de información" que corre a lo largo de la secuencia.

### Los 4 gates: controlando el flujo de información

La LSTM usa **gates** para controlar qué información entra, se mantiene, y sale de las memorias. Cada gate es un vector de valores entre 0 y 1 (gracias a sigmoid), donde 0 = "bloqueado" y 1 = "pasa todo".

#### Gate 1: Forget gate — "¿Qué olvidar del cell state?"

$$\mathbf{f}_t = \sigma(\mathbf{W}_{xf} \cdot \mathbf{x}_t + \mathbf{W}_{hf} \cdot \mathbf{h}_{t-1} + \mathbf{b}_f)$$

Mira el input actual y el hidden state anterior, y decide para cada dimensión del cell state: ¿la mantenemos o la olvidamos? Si $f_t^{(i)} \approx 1$, la dimensión $i$ del cell state se mantiene. Si $f_t^{(i)} \approx 0$, se olvida.

#### Gate 2: Input gate — "¿Cuánta información nueva agregar?"

$$\mathbf{i}_t = \sigma(\mathbf{W}_{xi} \cdot \mathbf{x}_t + \mathbf{W}_{hi} \cdot \mathbf{h}_{t-1} + \mathbf{b}_i)$$

Decide cuánta de la información nueva (candidata) va a entrar en el cell state. De nuevo, valores entre 0 (no agregar nada) y 1 (agregar todo).

#### Gate 3: Candidate — "¿Qué información nueva proponer?"

$$\mathbf{g}_t = \tanh(\mathbf{W}_{xg} \cdot \mathbf{x}_t + \mathbf{W}_{hg} \cdot \mathbf{h}_{t-1} + \mathbf{b}_g)$$

Nota que este usa **tanh** (no sigmoid). Genera valores entre -1 y 1, representando información nueva que **podría** agregarse al cell state. Pero no se agrega directamente — el input gate controla cuánto de esto entra realmente.

#### Gate 4: Output gate — "¿Qué exponer como hidden state?"

$$\mathbf{o}_t = \sigma(\mathbf{W}_{xo} \cdot \mathbf{x}_t + \mathbf{W}_{ho} \cdot \mathbf{h}_{t-1} + \mathbf{b}_o)$$

Decide qué partes del cell state (después de pasar por tanh) se exponen como output en el hidden state.

### Las ecuaciones de actualización

Con los 4 gates definidos, la actualización es:

**Actualizar cell state (memoria de largo plazo):**
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t$$

Esto se lee: "el nuevo cell state es lo que decidimos **mantener** del anterior ($\mathbf{f}_t \odot \mathbf{c}_{t-1}$) **más** la nueva información que decidimos **agregar** ($\mathbf{i}_t \odot \mathbf{g}_t$)". El $\odot$ es multiplicación elemento a elemento (Hadamard product).

**Actualizar hidden state (memoria de trabajo):**
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

El hidden state es una versión filtrada del cell state.

### ¿Por qué esto resuelve el vanishing gradient?

La clave está en la ecuación del cell state:

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t$$

El gradiente de $\mathbf{c}_t$ con respecto a $\mathbf{c}_{t-1}$ es simplemente $\mathbf{f}_t$ (más algunos términos menores). Si el forget gate está cerca de 1, el gradiente fluye **sin degradarse**. La red puede aprender a mantener el forget gate abierto para información que necesita recordar por mucho tiempo.

En contraste, en la RNN vanilla, el gradiente siempre pasa por tanh y $\mathbf{W}_{hh}$, lo que inevitablemente lo degrada.

In [ ]:
# LSTM implementation from scratch — step by step

class ManualLSTMCell:
    """LSTM cell implemented from scratch for educational purposes."""
    
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Initialize weights for all 4 gates
        # Each gate has: W_x (input weights), W_h (hidden weights), b (bias)
        scale = 1.0 / np.sqrt(hidden_dim)
        
        # Forget gate
        self.W_xf = torch.randn(hidden_dim, input_dim) * scale
        self.W_hf = torch.randn(hidden_dim, hidden_dim) * scale
        self.b_f = torch.ones(hidden_dim)  # bias init to 1 (keep memory by default!)
        
        # Input gate
        self.W_xi = torch.randn(hidden_dim, input_dim) * scale
        self.W_hi = torch.randn(hidden_dim, hidden_dim) * scale
        self.b_i = torch.zeros(hidden_dim)
        
        # Candidate (cell gate)
        self.W_xg = torch.randn(hidden_dim, input_dim) * scale
        self.W_hg = torch.randn(hidden_dim, hidden_dim) * scale
        self.b_g = torch.zeros(hidden_dim)
        
        # Output gate
        self.W_xo = torch.randn(hidden_dim, input_dim) * scale
        self.W_ho = torch.randn(hidden_dim, hidden_dim) * scale
        self.b_o = torch.zeros(hidden_dim)
    
    def forward(self, x_t, h_prev, c_prev, verbose=False):
        """One LSTM step."""
        
        # Forget gate: what to erase from cell state
        f_t = torch.sigmoid(self.W_xf @ x_t + self.W_hf @ h_prev + self.b_f)
        
        # Input gate: how much new info to add
        i_t = torch.sigmoid(self.W_xi @ x_t + self.W_hi @ h_prev + self.b_i)
        
        # Candidate: what new info to propose
        g_t = torch.tanh(self.W_xg @ x_t + self.W_hg @ h_prev + self.b_g)
        
        # Output gate: what to expose
        o_t = torch.sigmoid(self.W_xo @ x_t + self.W_ho @ h_prev + self.b_o)
        
        # Update cell state (long-term memory)
        c_t = f_t * c_prev + i_t * g_t
        
        # Update hidden state (working memory / output)
        h_t = o_t * torch.tanh(c_t)
        
        if verbose:
            print(f"  Forget gate f_t:    {f_t.detach().numpy().round(3)}")
            print(f"  Input gate i_t:     {i_t.detach().numpy().round(3)}")
            print(f"  Candidate g_t:      {g_t.detach().numpy().round(3)}")
            print(f"  Output gate o_t:    {o_t.detach().numpy().round(3)}")
            print(f"  Cell state c_t:     {c_t.detach().numpy().round(3)}")
            print(f"  Hidden state h_t:   {h_t.detach().numpy().round(3)}")
        
        return h_t, c_t, {'f': f_t, 'i': i_t, 'g': g_t, 'o': o_t}


# Run LSTM step by step
input_dim = 4
hidden_dim = 3
seq_len = 5

lstm_cell = ManualLSTMCell(input_dim, hidden_dim)
x = torch.randn(seq_len, input_dim)

# Initial states
h = torch.zeros(hidden_dim)
c = torch.zeros(hidden_dim)

print("LSTM Forward Pass — Step by Step")
print("=" * 60)

all_gates = []
for t in range(seq_len):
    print(f"\n--- Time step t={t} ---")
    print(f"  Input x_{t}: {x[t].numpy().round(3)}")
    h, c, gates = lstm_cell.forward(x[t], h, c, verbose=True)
    all_gates.append(gates)

print("\n" + "=" * 60)
print(f"\nFinal hidden state: {h.detach().numpy().round(3)}")
print(f"Final cell state:   {c.detach().numpy().round(3)}")
print("\nTwo types of memory: h (working) and c (long-term)!")

In [ ]:
# Visualize gate activations across time steps

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
gate_names = ['f', 'i', 'g', 'o']
gate_labels = ['Forget gate (f)', 'Input gate (i)', 'Candidate (g)', 'Output gate (o)']
gate_cmaps = ['Reds', 'Greens', 'coolwarm', 'Blues']
gate_ranges = [(0, 1), (0, 1), (-1, 1), (0, 1)]  # sigmoid vs tanh ranges

for ax, name, label, cmap, (vmin, vmax) in zip(axes.flat, gate_names, gate_labels, gate_cmaps, gate_ranges):
    data = np.array([all_gates[t][name].detach().numpy() for t in range(seq_len)])
    im = ax.imshow(data.T, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xlabel('Time step')
    ax.set_ylabel('Hidden dimension')
    ax.set_title(label, fontsize=13)
    ax.set_xticks(range(seq_len))
    plt.colorbar(im, ax=ax)

plt.suptitle('LSTM Gate Activations Over Time', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each gate dynamically controls information flow at each time step.")
print("Forget ≈ 1: keep memory. Input ≈ 1: add new info. Output ≈ 1: expose state.")

In [ ]:
# Compare gradient flow: RNN vs LSTM

def track_gradients(model_type, seq_len=50, hidden_dim=32, input_dim=16):
    """Track gradient norms for RNN or LSTM."""
    
    if model_type == 'rnn':
        model = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, batch_first=True)
    else:
        model = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, batch_first=True)
    
    x = torch.randn(1, seq_len, input_dim, requires_grad=True)
    
    if model_type == 'rnn':
        output, _ = model(x)
    else:
        output, _ = model(x)
    
    loss = output[0, -1, :].sum()
    loss.backward()
    
    grad_norms = []
    for t in range(seq_len):
        grad_norm = x.grad[0, t, :].norm().item()
        grad_norms.append(grad_norm)
    
    return grad_norms

# Average over multiple runs
seq_len = 60
rnn_norms_all = []
lstm_norms_all = []

for seed in range(30):
    torch.manual_seed(seed)
    rnn_norms_all.append(track_gradients('rnn', seq_len=seq_len))
    torch.manual_seed(seed)
    lstm_norms_all.append(track_gradients('lstm', seq_len=seq_len))

rnn_avg = np.mean(rnn_norms_all, axis=0)
lstm_avg = np.mean(lstm_norms_all, axis=0)

# Normalize to compare shapes
rnn_normalized = rnn_avg / rnn_avg.max()
lstm_normalized = lstm_avg / lstm_avg.max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute norms
ax = axes[0]
ax.semilogy(rnn_avg, 'o-', color='C3', markersize=3, label='RNN', alpha=0.8)
ax.semilogy(lstm_avg, 's-', color='C0', markersize=3, label='LSTM', alpha=0.8)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Gradient norm (log scale)', fontsize=12)
ax.set_title('Gradient Norms: RNN vs LSTM', fontsize=13)
ax.legend(fontsize=12)

# Normalized
ax = axes[1]
ax.plot(rnn_normalized, 'o-', color='C3', markersize=3, label='RNN', alpha=0.8)
ax.plot(lstm_normalized, 's-', color='C0', markersize=3, label='LSTM', alpha=0.8)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Normalized gradient', fontsize=12)
ax.set_title('Normalized Gradient Flow', fontsize=13)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"RNN:  gradient at t=0 is {rnn_normalized[0]:.6f} of max (step {np.argmax(rnn_avg)})")
print(f"LSTM: gradient at t=0 is {lstm_normalized[0]:.6f} of max (step {np.argmax(lstm_avg)})")
print(f"\nLSTM maintains MUCH better gradient flow to early time steps!")

### Contando parámetros: RNN vs LSTM

La LSTM tiene **4 veces** más parámetros que una RNN vanilla, porque tiene 4 sets de pesos (uno por gate) donde la RNN tiene 1:

| Modelo | Parámetros |
|:-------|:-----------|
| **RNN** | $H \times D + H \times H + H$ (un set de $\mathbf{W}_{xh}$, $\mathbf{W}_{hh}$, $\mathbf{b}$) |
| **LSTM** | $4 \times (H \times D + H \times H + H)$ (cuatro sets: forget, input, candidate, output) |

Para $D = 256$, $H = 512$:
- RNN: $512 \times 256 + 512 \times 512 + 512 = 393,\!728$ parámetros
- LSTM: $4 \times 393,\!728 = 1,\!574,\!912$ parámetros

Más parámetros, pero cada uno tiene un **rol funcional específico**. No son parámetros "extra" por tirarlos — cada gate tiene un propósito claro.

---

<a id='intuicion-diseno'></a>
## 5. Intuición de diseño de arquitecturas

### Esta sección es MUY importante

Vamos a parar acá un momento y reflexionar sobre algo profundo. Lo que hicieron Hochreiter y Schmidhuber con la LSTM no fue simplemente "agregar más neuronas". Lo que hicieron fue **diseñar la lógica estructural del sistema**.

Pensalo así:

### Diseñar arquitecturas es modelar lógica estructural

Cuando diseñás una arquitectura de red neuronal, lo que estás haciendo es:

1. **Asignar roles a distintos grupos de pesos**
2. **Definir cómo interactúan esos grupos a través de las ecuaciones**
3. **Dejar que el entrenamiento determine los valores específicos**

En la LSTM:
- Los pesos del forget gate "aprenden" qué información borrar
- Los pesos del input gate "aprenden" cuánta info nueva aceptar
- Los pesos del candidate "aprenden" qué info nueva proponer
- Los pesos del output gate "aprenden" qué exponer

**El significado funcional de cada grupo de pesos lo define la posición que ocupa en las ecuaciones.** Los mismos números (pesos) significan cosas totalmente distintas dependiendo de dónde los pongas en la arquitectura.

### Los pesos aprenden "cuánto", nosotros definimos "qué tipo de operación"

Cuando diseñamos la LSTM, no le dijimos a la red cuánto olvidar o cuánto recordar. Eso lo aprende sola durante el entrenamiento. Lo que **sí** definimos es:

- Que **existe una operación de olvido** (forget gate)
- Que **existe una operación de filtrado de entrada** (input gate)
- Que **existe una memoria protegida** (cell state)
- Que **la memoria se actualiza aditivamente** ($c_t = f_t \odot c_{t-1} + i_t \odot g_t$)

"Solo le damos la estructura". La red llena los valores.

### Ejemplo concreto: GRU (Gated Recurrent Unit)

Para ver este principio en acción, mirá la **GRU** (Cho et al., 2014), que es como una LSTM simplificada:

![GRU (Gated Recurrent Unit)](../ai_notas/AI%20notas/image%2052.png)

La GRU combina el forget gate y el input gate en un solo "update gate" $\mathbf{z}_t$, y elimina el cell state separado. Tiene menos parámetros que la LSTM pero funciona sorprendentemente bien en muchas tareas.

$$\mathbf{z}_t = \sigma(\mathbf{W}_{xz} \cdot \mathbf{x}_t + \mathbf{W}_{hz} \cdot \mathbf{h}_{t-1} + \mathbf{b}_z)$$
$$\mathbf{r}_t = \sigma(\mathbf{W}_{xr} \cdot \mathbf{x}_t + \mathbf{W}_{hr} \cdot \mathbf{h}_{t-1} + \mathbf{b}_r)$$
$$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_{xh} \cdot \mathbf{x}_t + \mathbf{W}_{hh} \cdot (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_h)$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

Fijate en la última ecuación: es elegantísima. El update gate $\mathbf{z}_t$ interpola entre mantener el hidden anterior ($1 - \mathbf{z}_t$) y usar el nuevo candidato ($\mathbf{z}_t$). **Una sola perilla controla el balance entre recordar y actualizar.**

Distinta estructura → distinto "tipo de pensamiento" → misma idea de diseño: **asignar roles a pesos a través de las ecuaciones**.

---

<a id='turing-complete'></a>
## 6. RNNs son Turing-Complete (en teoría)

### Un resultado teórico fascinante

Las RNNs son **Turing-complete** (Siegelmann & Sontag, 1995). ¿Qué significa esto? Que en teoría, una RNN con suficientes neuronas puede computar **cualquier función computable**. Es decir, puede simular cualquier programa de computadora.

¿Por qué? Pensá en lo que hace una RNN en cada paso:

1. **Recibe un input** $\mathbf{x}_t$ (como leer de un tape)
2. **Tiene un estado interno** $\mathbf{h}_t$ (como las variables de un programa)
3. **Aplica una función fija pero aprendida** para generar un nuevo estado:
   $$\mathbf{h}_t = f(\mathbf{x}_t, \mathbf{h}_{t-1}; \theta)$$

Esto es exactamente lo que hace un programa:
- Lee inputs
- Tiene variables internas (estado)
- Aplica instrucciones (función fija) para actualizar las variables

La función $f$ es siempre la misma (los pesos no cambian durante inference), pero al haber sido **aprendida** durante entrenamiento, puede implementar lógica arbitrariamente compleja.

### Limitaciones prácticas

En la práctica, las RNNs están lejos de ser computadoras universales:
- El hidden state tiene **dimensión finita** (mientras que una máquina de Turing tiene tape infinito)
- El vanishing gradient limita la profundidad de la "computación" efectiva
- Entrenarlas para tareas algorítmicas complejas es extremadamente difícil

Pero el resultado teórico es importante porque nos dice que la **estructura** de "combinar input con state usando una función aprendida" es fundamentalmente poderosa. Esta idea reaparece en muchas arquitecturas modernas.

---

<a id='deep-rnn'></a>
## 7. Deep RNN (RNNs apiladas)

### ¿Por qué apilar capas?

Igual que con los MLPs, podemos hacer RNNs más profundas **apilando capas**. La idea es que cada capa aprende un nivel diferente de abstracción:

- **Capa 1**: patrones de bajo nivel (ej: combinaciones de letras, fonemas)
- **Capa 2**: patrones de nivel medio (ej: palabras, frases cortas)
- **Capa 3**: patrones de alto nivel (ej: temas, sentimiento, intención)

### Cómo funciona

En una Deep RNN, el hidden state de cada capa sirve como input para la capa siguiente. Además, cada capa mantiene su propio hidden state que se pasa al siguiente paso temporal.

Es decir, el hidden state va en **dos direcciones**:
1. **Hacia arriba** → a la siguiente capa (como input)
2. **Hacia la derecha** → al siguiente paso temporal (como memoria)

![Deep RNN - Stacking layers](../ai_notas/AI%20notas/image%2053.png)

![Deep RNN - Information flow](../ai_notas/AI%20notas/image%2054.png)

![Deep RNN - Multiple layers with connections](../ai_notas/AI%20notas/image%2055.png)

In [ ]:
# Demo: Deep RNN with PyTorch

input_dim = 16
hidden_dim = 32
num_layers = 3
seq_len = 10
batch_size = 1

# Single-layer vs multi-layer LSTM
lstm_1layer = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                      num_layers=1, batch_first=True)
lstm_3layer = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                      num_layers=3, batch_first=True)

x = torch.randn(batch_size, seq_len, input_dim)

# Forward pass
out_1, (h_1, c_1) = lstm_1layer(x)
out_3, (h_3, c_3) = lstm_3layer(x)

print("Single-layer LSTM:")
print(f"  Output shape: {out_1.shape}  (batch, seq_len, hidden_dim)")
print(f"  Hidden shape: {h_1.shape}  (num_layers, batch, hidden_dim)")
print(f"  Cell shape:   {c_1.shape}  (num_layers, batch, hidden_dim)")
print(f"  Parameters:   {sum(p.numel() for p in lstm_1layer.parameters()):,}")

print(f"\n3-layer Deep LSTM:")
print(f"  Output shape: {out_3.shape}  (batch, seq_len, hidden_dim) — output of LAST layer")
print(f"  Hidden shape: {h_3.shape}  (num_layers, batch, hidden_dim) — one h per layer")
print(f"  Cell shape:   {c_3.shape}  (num_layers, batch, hidden_dim) — one c per layer")
print(f"  Parameters:   {sum(p.numel() for p in lstm_3layer.parameters()):,}")

print(f"\nNote: 3-layer has {sum(p.numel() for p in lstm_3layer.parameters()) / sum(p.numel() for p in lstm_1layer.parameters()):.1f}x more parameters")
print("The hidden state flows both UP (to next layer) and RIGHT (to next time step)")

En la práctica, 2-4 capas suelen ser suficientes para la mayoría de las tareas. Más capas hacen el entrenamiento más difícil sin necesariamente mejorar el rendimiento (salvo que tengas datasets enormes).

Google's Neural Machine Translation (2016) usaba 8 capas de LSTM — y eso ya era considerado "muy profundo" para la época. El paper reportaba que las capas superiores se beneficiaban de residual connections para facilitar el entrenamiento.

---

<a id='bottleneck'></a>
## 8. El bottleneck del hidden state

### El mayor problema de las RNNs (incluso LSTM)

Ok, la LSTM resuelve el vanishing gradient. Genial. Pero hay un problema más fundamental que ni la LSTM ni la GRU resuelven: **toda la memoria de la secuencia está comprimida en un solo vector**.

Pensalo: leíste un libro entero de 100,000 palabras. Toda la información sobre ese libro tiene que caber en un vector $\mathbf{h}_T$ de dimensión $H$ (típicamente 256, 512, o 1024). Eso es como intentar resumir un libro en un tweet.

### El problema de la compresión excesiva

$$\underbrace{\mathbf{x}_1, \mathbf{x}_2, \mathbf{x}_3, \ldots, \mathbf{x}_T}_{\text{secuencia completa}} \xrightarrow{\text{RNN/LSTM}} \underbrace{\mathbf{h}_T}_{\text{un solo vector de dim } H}$$

El hidden state final $\mathbf{h}_T$ tiene que codificar **toda** la información relevante de la secuencia. Pero un vector de dimensión fija tiene capacidad limitada. Cuanto más larga es la secuencia, más información tiene que comprimir en el mismo espacio.

### No se puede acceder directamente a distintas partes del input

El otro problema es que con una RNN, si querés saber algo sobre el token 5, no podés "volver" y mirarlo. Solo tenés acceso al hidden state final, que es un resumen comprimido de todo. No hay forma de decir "necesito más detalle sobre la parte del input que hablaba de X".

Esto es como si alguien te contara una historia larga y después vos tuvieras que responder preguntas, pero **no pudieras releer la historia** — solo pudieras usar tu memoria de lo que escuchaste.

### Implicaciones

Este bottleneck tiene consecuencias prácticas muy reales:

1. **El rendimiento se degrada con secuencias largas**: cuanto más larga la secuencia, más información se pierde por la compresión

2. **Bias hacia los últimos tokens**: la información de los últimos tokens está "más fresca" en el hidden state. Los primeros tokens tienden a ser sobreescritos

3. **No hay acceso selectivo**: no podés buscar información específica en la secuencia — solo tenés el resumen comprimido

Para resolver este problema de verdad, necesitamos un mecanismo que permita al modelo **mirar directamente cualquier parte del input cuando lo necesite**. Eso es exactamente lo que hace **attention**, que veremos en el próximo notebook.

---

<a id='seq2seq'></a>
## 9. Seq2Seq (Encoder-Decoder)

### El problema de sequence-to-sequence

Hay muchas tareas donde tanto el input como el output son secuencias, pero de **longitudes diferentes**:

| Tarea | Input | Output |
|:------|:------|:-------|
| **Traducción** | "el gato se sentó" (4 tokens) | "the cat sat down" (4 tokens, pero podría ser distinto) |
| **Resumen** | Un artículo de 500 palabras | Un resumen de 50 palabras |
| **Diálogo** | "¿Cómo estás?" (2 tokens) | "Bien, gracias por preguntar" (4 tokens) |
| **Speech-to-text** | Audio de 10 segundos | Texto de 15 palabras |

No podemos usar una sola RNN porque el input y el output pueden tener longitudes distintas.

### La arquitectura encoder-decoder

La solución (Sutskever et al., 2014) es usar **dos RNNs** (en la práctica, dos LSTMs):

1. **Encoder**: procesa la secuencia de entrada completa y produce un hidden state final $\mathbf{h}_{enc}$. Este vector es el "resumen" de todo el input.

2. **Decoder**: recibe $\mathbf{h}_{enc}$ como su hidden state inicial y genera la secuencia de salida **un token a la vez**, autoregresivamente.

```
ENCODER:                              DECODER:
  x_1  x_2  x_3  x_4  <EOS>          <BOS>  y_1   y_2   y_3
   │    │    │    │     │               │     │     │     │
   ▼    ▼    ▼    ▼     ▼               ▼     ▼     ▼     ▼
  ┌──┐ ┌──┐ ┌──┐ ┌──┐ ┌──┐           ┌──┐  ┌──┐  ┌──┐  ┌──┐
  │h1│→│h2│→│h3│→│h4│→│h5│ ========▶ │h1│→ │h2│→ │h3│→ │h4│
  └──┘ └──┘ └──┘ └──┘ └──┘   h_enc   └──┘  └──┘  └──┘  └──┘
                                        │     │     │     │
                                        ▼     ▼     ▼     ▼
                                       y_1   y_2   y_3  <EOS>
```

### Generación autoregresiva

El decoder genera un token a la vez:
1. Recibe el token especial `<BOS>` (beginning of sequence) y $\mathbf{h}_{enc}$
2. Produce una distribución de probabilidad sobre el vocabulario → selecciona el primer token $y_1$
3. Alimenta $y_1$ como input al siguiente paso → produce $y_2$
4. ... continúa hasta generar `<EOS>` (end of sequence) o llegar a un largo máximo

Cada paso es:
$$\mathbf{h}_t^{dec} = \text{LSTM}(y_{t-1}, \mathbf{h}_{t-1}^{dec})$$
$$P(y_t | y_{<t}, \mathbf{x}) = \text{softmax}(\mathbf{W}_{out} \cdot \mathbf{h}_t^{dec} + \mathbf{b}_{out})$$

In [ ]:
# Complete Seq2Seq model with LSTM

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
    
    def forward(self, x):
        # x: (batch, seq_len) — token indices
        embedded = self.embedding(x)  # (batch, seq_len, embed_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # hidden, cell: (num_layers, batch, hidden_dim)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden, cell):
        # x: (batch, 1) — single token
        embedded = self.embedding(x)  # (batch, 1, embed_dim)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: (batch, 1, hidden_dim)
        logits = self.fc_out(output.squeeze(1))  # (batch, vocab_size)
        return logits, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device='cpu'):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (batch, src_len) — input sequence
        # trg: (batch, trg_len) — target sequence
        
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        vocab_size = self.decoder.fc_out.out_features
        
        # Store outputs
        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)
        
        # Encode — get the final hidden state (the "summary")
        hidden, cell = self.encoder(src)
        
        # First input to decoder is <BOS> token (index 1)
        input_token = trg[:, 0:1]  # (batch, 1)
        
        # Decode one token at a time
        for t in range(1, trg_len):
            logits, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t, :] = logits
            
            # Teacher forcing: use real target vs predicted token
            if np.random.random() < teacher_forcing_ratio:
                input_token = trg[:, t:t+1]  # use real target
            else:
                input_token = logits.argmax(dim=1, keepdim=True)  # use prediction
        
        return outputs
    
    def generate(self, src, bos_token=1, eos_token=2, max_len=50):
        """Autoregressive generation (inference)."""
        self.eval()
        with torch.no_grad():
            # Encode
            hidden, cell = self.encoder(src)
            
            # Start with <BOS>
            input_token = torch.tensor([[bos_token]]).to(self.device)
            generated = [bos_token]
            
            for _ in range(max_len):
                logits, hidden, cell = self.decoder(input_token, hidden, cell)
                next_token = logits.argmax(dim=1).item()
                generated.append(next_token)
                
                if next_token == eos_token:
                    break
                
                input_token = torch.tensor([[next_token]]).to(self.device)
        
        self.train()
        return generated


# Create a small Seq2Seq model
vocab_size = 100
embed_dim = 32
hidden_dim = 64

encoder = Encoder(vocab_size, embed_dim, hidden_dim)
decoder = Decoder(vocab_size, embed_dim, hidden_dim)
model = Seq2Seq(encoder, decoder)

print("Seq2Seq Model Architecture:")
print(f"  Vocab size:   {vocab_size}")
print(f"  Embed dim:    {embed_dim}")
print(f"  Hidden dim:   {hidden_dim}")
print(f"  Encoder params: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  Decoder params: {sum(p.numel() for p in decoder.parameters()):,}")
print(f"  Total params:   {sum(p.numel() for p in model.parameters()):,}")

# Demo forward pass
src = torch.randint(0, vocab_size, (1, 8))   # batch of 1, source length 8
trg = torch.randint(0, vocab_size, (1, 6))   # batch of 1, target length 6

print(f"\nForward pass:")
print(f"  Source shape: {src.shape}  (batch=1, src_len=8)")
print(f"  Target shape: {trg.shape}  (batch=1, trg_len=6)")

output = model(src, trg)
print(f"  Output shape: {output.shape}  (batch=1, trg_len=6, vocab_size=100)")

# Demo generation
generated = model.generate(src, bos_token=1, eos_token=2, max_len=10)
print(f"\nGenerated sequence: {generated}")
print("(Random tokens since the model is not trained — but the pipeline works!)")

### El bottleneck en Seq2Seq

Fijate algo crucial: **toda la información del input se pasa al decoder a través de un solo vector** ($\mathbf{h}_{enc}$ y $\mathbf{c}_{enc}$). Ese vector es el hidden state final del encoder.

Esto funciona razonablemente bien para secuencias cortas, pero para secuencias largas, es un cuello de botella brutal. Si estás traduciendo un párrafo de 200 palabras, toda la información de esas 200 palabras tiene que comprimirse en un vector de dimensión 512.

```
200 palabras ──────────▶ 1 vector (dim 512) ──────────▶ traducción completa
                              bottleneck!
```

En la práctica, los modelos Seq2Seq vanilla funcionan bien para oraciones de hasta ~20-30 tokens pero se degradan significativamente para secuencias más largas.

La solución a este bottleneck — el **mecanismo de attention** — es una de las ideas más importantes en la historia del deep learning, y la veremos en detalle en el próximo notebook.

In [ ]:
# Demo: how the hidden state struggles with longer sequences
# We'll measure the reconstruction capacity of the encoder's hidden state

class SimpleAutoencoder(nn.Module):
    """Encode a sequence into a hidden state, then try to reconstruct it."""
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        embedded = self.embedding(x)
        _, (h, c) = self.encoder(embedded)
        # Decode using the same embeddings but starting from encoder's final state
        output, _ = self.decoder(embedded, (h, c))
        logits = self.fc(output)
        return logits

vocab_size = 50
embed_dim = 16
hidden_dim = 32

results = {}
for seq_len in [5, 10, 20, 40, 80]:
    model = SimpleAutoencoder(vocab_size, embed_dim, hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Train on random sequences
    losses = []
    for step in range(300):
        x = torch.randint(0, vocab_size, (16, seq_len))  # batch of 16
        logits = model(x)  # (batch, seq_len, vocab_size)
        
        loss = F.cross_entropy(logits.view(-1, vocab_size), x.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    # Measure accuracy on new sequences
    model.eval()
    with torch.no_grad():
        x = torch.randint(0, vocab_size, (100, seq_len))
        logits = model(x)
        preds = logits.argmax(dim=-1)
        accuracy = (preds == x).float().mean().item()
    
    results[seq_len] = {'loss': losses[-1], 'accuracy': accuracy}
    print(f"Seq len {seq_len:3d}: final loss = {losses[-1]:.3f}, accuracy = {accuracy:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
lengths = list(results.keys())
accuracies = [results[l]['accuracy'] for l in lengths]

ax.bar(range(len(lengths)), accuracies, color=['#2ecc71' if a > 0.5 else '#e74c3c' for a in accuracies])
ax.set_xticks(range(len(lengths)))
ax.set_xticklabels([str(l) for l in lengths])
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Reconstruction Accuracy', fontsize=12)
ax.set_title('Hidden State Bottleneck:\nReconstruction accuracy degrades with sequence length', fontsize=13)
ax.set_ylim(0, 1.05)
for i, acc in enumerate(accuracies):
    ax.text(i, acc + 0.02, f'{acc:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print("\nAs sequences get longer, the fixed-size hidden state can't encode all the info.")
print("This is the fundamental bottleneck that ATTENTION will solve.")

---

<a id='resumen'></a>
## 10. Resumen

### De secuencias fijas a secuencias variables

En este notebook recorrimos la evolución de las redes neuronales para procesar secuencias:

| Concepto | Descripción | Limitación principal |
|:---------|:------------|:---------------------|
| **MLP + padding** | Flatten toda la secuencia en un vector fijo | No escala, no comparte pesos, pierde estructura temporal |
| **RNN vanilla** | Procesa un token a la vez, mantiene hidden state como memoria | Vanishing gradients → no puede aprender dependencias de largo plazo |
| **LSTM** | Dos memorias (h y c) + gates para controlar flujo de información | Resuelve vanishing gradient, pero toda la memoria sigue siendo un solo vector |
| **GRU** | Versión simplificada de LSTM (2 gates en vez de 4) | Similar a LSTM, a veces mejor con menos datos |
| **Deep RNN** | Múltiples capas de RNN/LSTM apiladas | Más expresivo, pero más difícil de entrenar |
| **Seq2Seq** | Encoder-decoder para mapear secuencia → secuencia | El hidden state final es un bottleneck para secuencias largas |

### Ideas clave para recordar

1. **Weight sharing** es lo que hace que las RNNs funcionen: los mismos pesos procesan cada paso temporal
2. **El diseño de arquitecturas es asignar roles a grupos de pesos** a través de las ecuaciones
3. **Los gates de la LSTM** son la primera instancia de "mecanismos de control aprendidos" en deep learning
4. **El hidden state como bottleneck** es el problema fundamental que motiva el mecanismo de attention
5. **Las RNNs procesan secuencialmente** (un paso a la vez), lo que las hace lentas para entrenar con secuencias largas

### ¿Qué viene después?

El próximo paso es resolver el bottleneck del hidden state. La idea es simple pero revolucionaria: **en vez de comprimir toda la secuencia en un vector, dejar que el decoder mire directamente cualquier parte del input cuando lo necesite**. Eso es el mecanismo de **attention**, y es la base de los Transformers que dominan el deep learning hoy.

---

**Siguiente notebook →** [10 - Attention](./10_attention.ipynb): el mecanismo que permite mirar directamente cualquier parte del input, resolviendo el bottleneck del hidden state y sentando las bases para los Transformers.